What is a validation retry loop in the context of structured output extraction?
A.A loop that automatically retries the API call three times on any error regardless of its cause
B.A client-side retry mechanism for handling API rate limits gracefully
C.A mechanism that validates user input before sending it to the Claude API
D.A pattern where the model gets its validation errors and regenerates until passing
✗ Incorrect — Explanation
Validation retry loops are a self-healing pattern: (1) request structured output, (2) validate against schema, (3) if validation fails, send the specific error back to Claude ("Field `price` must be a number, you returned a string"), (4) Claude regenerates with that specific feedback. Without the error feedback, Claude cannot self-correct reliably.


A pull request modifies 14 files across the stock tracking module. Your single-pass review analyzing all files together produces inconsistent results: detailed feedback for some files but superficial comments for others, obvious bugs missed, and contradictory feedback — flagging a pattern as problematic in one file while approving identical code elsewhere in the same PR. How should you restructure the review?
A.Split into focused passes: analyze each file individually for local issues, then run a separate integration-focused pass examining cross-file data flow
B.Require developers to split large PRs into smaller submissions of 3–4 files before the automated review runs
C.Switch to a higher-tier model with a larger context window to give all 14 files adequate attention in one pass
D.Run three independent review passes on the full PR and only flag issues that appear in at least two of the three runs
✓ Correct — Explanation
Splitting reviews into focused passes directly addresses the root cause: attention dilution when processing many files at once. File-by-file analysis ensures consistent depth, while a separate integration pass catches cross-file issues. Option B shifts burden to developers without improving the system. Option C misunderstands that larger context windows don't solve attention quality issues. Option D would actually suppress detection of real bugs by requiring consensus on issues that may only be caught intermittently.


Your automated review analyzes comments and docstrings. The current prompt instructs Claude to "check that comments are accurate and up-to-date." Findings frequently flag acceptable patterns (TODO markers, straightforward descriptions) while missing comments that describe behavior the code no longer implements. What change addresses the root cause of this inconsistent analysis?
A.Include git blame data so Claude can identify comments that predate recent code modifications
B.Filter out TODO, FIXME, and descriptive comment patterns before analysis to reduce noise
C.Add few-shot examples of misleading comments to help the model recognize similar patterns in the codebase
D.Specify explicit criteria: flag comments only when their claimed behavior contradicts actual code behavior
✗ Incorrect — Explanation
The root cause is the vague instruction "check that comments are accurate and up-to-date," which provides no concrete definition of what makes a comment problematic. Specifying explicit criteria — flag a comment only when its stated behavior contradicts what the code actually does — removes the ambiguity that generates both false positives (TODO markers, obvious descriptions) and false negatives (stale behavioral claims). This directly targets the mismatch between claim and implementation. Git blame (A) adds noise without a decision rule. Filtering (B) is a workaround, not a fix. Few-shot examples (C) help but are secondary to fixing the missing criterion.



Your code review pipeline produces structured findings that developers frequently dismiss as false positives. You want to systematically improve the pipeline's precision over time. Which schema addition enables this analysis?
A.Add a `severity: "critical" | "high" | "medium" | "low"` field to filter out low-severity findings before display
B.Add a `detected_pattern` field to each finding that names the code construct that triggered the finding (e.g., `"eval_in_userland"`, `"unvalidated_redirect"`); when developers dismiss findings, analyze which patterns are dismissed most frequently to identify false positive candidates
C.Add a `model_confidence: number` field and only show findings where confidence > 0.8
D.Add a `fix_suggestion: string` field so developers can accept the fix with one click instead of dismissing
✓ Correct — Explanation
The `detected_pattern` field instruments findings with the code construct that triggered them. When developers dismiss a finding, that pattern name is logged. Over time, patterns that are dismissed at high rates (e.g., 80% of `"eval_in_userland"` findings are dismissed in this codebase) are identified as false positive candidates for prompt tuning or rule exclusion. Severity filtering (A) reduces noise but does not provide signal for improvement. Model confidence scores (C) are poorly calibrated — high confidence does not correlate reliably with correctness. Fix suggestions (D) improve developer experience but do not help identify systematic false positive patterns.


You are designing a prompt for a CI code review tool that must provide actionable, specific feedback and minimize false positives. Which prompt engineering technique is MOST important?
A.Chain-of-thought: ask Claude to reason step by step
B.Few-shot: provide examples of good vs. bad review comments, distinguishing true positives from common false positives specific to your codebase
C.Higher temperature: generate more diverse feedback
D.Longer prompts: include all possible coding standards in the system prompt
✓ Correct — Explanation
Few-shot examples are uniquely powerful for CI review because they demonstrate the style distinction between actionable feedback ("Line 47: `user_input` is used in an SQL query without parameterization") vs. vague false positives ("This code might have security issues"). Pairing positive examples (good reviews) with negative examples (common false positives in your codebase) calibrates Claude's judgment to your team's standards.


Your extraction system processes invoices and must populate 8 mandatory fields. On the first pass, 18% of invoices fail validation — mostly because `currency_code` is returned as "US Dollars" instead of the ISO 4217 code "USD". After implementing a retry loop, failures drop to 2%. The remaining 2% have ambiguous or missing currency information in the source document. How should you handle the persistent 2%?
A.Run them through three more retry passes until they succeed
B.Use a default currency code of "USD" for all remaining failures
C.Mark them with a `requires_human_review: true` flag and `confidence: "low"` field, route to a manual review queue rather than auto-processing
D.Drop them from the pipeline to maintain data quality
✓ Correct — Explanation
When repeated retry passes still fail, the underlying issue is ambiguous source data — no amount of retrying will produce a reliable answer. The correct architectural pattern is graceful degradation: set `requires_human_review: true` and `confidence: "low"`, route to a manual review queue, and process the remaining 98% automatically. This maximizes automation while maintaining data quality for the edge cases that genuinely need human judgment.



Few-shot examples are effective for which purpose, and NOT effective for which purpose?
A.Effective for format and style demonstration; not for guaranteeing compliance
B.Effective for compliance enforcement; not effective for output format control
C.Effective for preventing hallucinations; not effective for formatting output
D.Effective for all purposes universally; no known limitations for few-shot use
✓ Correct — Explanation
Few-shot examples are excellent for demonstrating desired output format, style, and reasoning patterns — Claude reliably mimics demonstrated patterns. They are NOT compliance mechanisms: showing Claude examples of correct tool ordering does not guarantee it will always follow that order (a probabilistic influence, not deterministic enforcement). For compliance, use programmatic mechanisms.


You are building a document enrichment pipeline. Documents must first go through metadata extraction (extract_metadata tool) before any enrichment steps run. If metadata extraction is skipped, enrichment tools may fail or produce incorrect results. How do you enforce this ordering using the Claude API?
A.Add a system prompt instruction: "Always call extract_metadata before any enrichment tools run"
B.Use tool_choice: "any" so Claude is guaranteed to call a tool on the first turn
C.Set tool_choice to force extract_metadata on the first call; use auto for the rest
D.List extract_metadata first in the tools array so Claude encounters it first
✗ Incorrect — Explanation
Forced tool selection (tool_choice: {"type": "tool", "name": "extract_metadata"}) guarantees that the specific named tool is called, not just any tool. By using this on the first API call, you ensure metadata extraction runs before the model has the option to call enrichment tools. Subsequent turns can then use tool_choice: "auto" or "any" for the enrichment steps. System prompt instructions (A) are probabilistic. tool_choice: "any" (C) guarantees a tool call but doesn't specify which one. Tool array ordering (D) has no effect on tool selection.



Your CI pipeline generates code review comments using Claude. Developers complain that 40% of security-related comments are false positives about their use of `eval()` in a controlled, sandboxed context. You have documented that `eval()` in the `/sandbox` directory is reviewed and approved. What is the most targeted fix?
A.Remove all security checks from the automated review
B.Add a post-processing filter that removes any comment mentioning `eval()`
C.Add to CLAUDE.md: "In the /sandbox directory, eval() usage has been security-reviewed and approved. Do not flag eval() in /sandbox as a security issue."
D.Increase the review model's temperature to reduce repeated patterns
✓ Correct — Explanation
This is a targeted CLAUDE.md context injection. The false positives are caused by Claude lacking project-specific knowledge that `/sandbox/eval()` is approved. Adding this specific exception to CLAUDE.md tells Claude exactly which pattern to exempt and why — without removing eval() coverage elsewhere. Removing all security checks or adding blanket filters are over-corrections that reduce security coverage across the codebase.



A product classification pipeline achieves 94% accuracy on well-formatted product listings but drops to 71% on short, abbreviated listings (e.g., "16GB DDR5 3200MHz" instead of "Kingston 16GB DDR5 RAM Module 3200MHz CL16"). How should you address this?
A.Filter out short listings and process them manually
B.Use a higher temperature for short listings to generate more diverse outputs
C.Add few-shot examples specifically demonstrating classification of abbreviated, incomplete listings, including the reasoning process for resolving ambiguous abbreviations
D.Use a larger model for all listings to improve overall accuracy
✓ Correct — Explanation
Few-shot examples are ideally suited for demonstrating how to handle the specific failure mode: abbreviated input. By showing Claude several examples of abbreviated listings and the reasoning used to classify them ("DDR5 3200MHz → RAM category, 3200MHz is the speed → subcategory: Memory"), you teach the pattern without modifying the schema. This is targeted prompt engineering for a specific input distribution.


When is it NOT appropriate to use the Message Batches API?
A.For classifying 100,000 support tickets overnight with results by morning
B.For generating weekly summary reports that run on Sunday each week
C.For live user queries in a chat app that requires sub-2-second latency
D.For bulk document analysis with results needed the next business day
✓ Correct — Explanation
The Message Batches API has a 24-hour processing window with no latency SLA. Live user queries requiring sub-2-second responses must use the synchronous API. Using Batches for live interactions would result in users waiting minutes or hours for responses. The key filter: if a real human is waiting for the answer right now, use synchronous API.


You are extracting information from legal contracts. Some contracts contain penalty clauses with specific dollar amounts; others don't mention penalties at all. Your extraction schema has a penalty_amount field. What is the correct way to define this field to prevent hallucination?
A.Define penalty_amount as a required string field with a default value of "N/A"
B.Omit penalty_amount from the schema and use a free-text notes field instead
C.Add an instruction: "If penalty_amount is not mentioned, set the value to 0"
D.Define penalty_amount as optional (nullable) so null signals genuine absence
✗ Incorrect — Explanation
Required fields create pressure on the model to produce a value even when none exists in the source document — leading to hallucinated data. Making penalty_amount optional (nullable) allows the model to return null when no penalty clause is present, which correctly signals absence rather than fabricating a value. Setting a default of "N/A" (A) is a string, not a sentinel for absence, and may be confused with actual contract text. A free-text notes field (C) loses structure. Defaulting to 0 (D) fabricates a value that may be mistaken for a real contract amount.


You are designing a JSON schema for a medical claims classification pipeline. The category field must be one of a predefined list, but new claim types occasionally appear that do not fit any category. What schema pattern prevents hallucinated category values while keeping the pipeline extensible?
A.Use a free-text `category` string field so the model can describe any claim type in its own words
B.Make `category` a required enum and retrain the model when new types appear
C.Add an `"other"` enum value with a companion `category_detail` field for specifics
D.Use a category array so the model can select multiple values when one won't fit
✓ Correct — Explanation
A free-text field (A) removes the guarantee of controlled vocabulary — the model will produce different strings for the same concept across runs. A pure enum without a catch-all (C) forces the model to pick the nearest wrong category rather than admitting the case does not fit, degrading accuracy. A multi-select array (D) changes the semantics of the field. The correct pattern is `enum: ["medical", "dental", "pharmacy", "other"]` paired with `category_detail: string | null` — populated only when `category === "other"`. This preserves controlled vocabulary for known types, enables downstream filtering and aggregation, and gives a structured escape hatch for novel cases that surfaces them for human review without hallucinating a false category.


The Message Batches API offers approximately what cost reduction compared to the standard synchronous API?
A.10%
B.25%
C.50%
D.75%
✓ Correct — Explanation
The Message Batches API offers approximately 50% cost savings compared to real-time API calls. The tradeoff is a 24-hour processing window with no SLA guarantees. This makes it suitable only for latency-tolerant, non-interactive workloads — batch classification, overnight analysis, bulk transformations.


Your extraction pipeline produces a result where line_items sum to $1,250 but the stated_total field is $1,200. Schema validation passes because both fields are present and correctly typed, but the values are semantically inconsistent. What is the correct approach for handling this in a retry loop?
A.Send the original document with the failed extraction and the specific semantic error for re-extraction
B.Accept the output without further checks — schema validation passed, so the extraction is considered correct by the pipeline
C.Set stated_total = sum(line_items) programmatically before passing it downstream
D.Retry the same request three times and use the most common value for stated_total
✗ Incorrect — Explanation
Semantic validation errors (inconsistent values) require retry-with-error-feedback: include the original document, the failed extraction, and a precise description of the error. The model can then re-examine the source document to determine whether stated_total or line_items is wrong. Option A is wrong — schema validity doesn't imply semantic correctness. Option C silently picks an answer without checking which value is correct in the source. Option D retries the same context without feedback, which rarely fixes semantic errors.


What is a few-shot prompt?
A.A prompt sent multiple times until Claude eventually produces the right answer through repetition
B.A prompt with input-output examples that demonstrate the desired behavior pattern
C.A prompt that limits Claude to short responses of only a few sentences each
D.A batch of multiple prompts that are processed simultaneously in one API call
✓ Correct — Explanation
Few-shot prompting provides examples of the desired input-output pattern within the prompt. These examples act as demonstrations — they show Claude the format, style, and reasoning pattern expected. Few-shot examples are effective for establishing output format and tone but are not compliance mechanisms (they cannot guarantee behavior).



What does `strict: true` on a tool definition guarantee?
A.Claude will always use this specific tool regardless of the conversation context or user intent
B.Tool calls will always match the defined JSON schema — no missing or wrong fields
C.The tool will reject invalid inputs at the MCP server level before processing
D.Claude will validate the tool's output before returning results to the user
✓ Correct — Explanation
`strict: true` enables Structured Outputs for tool use, guaranteeing that every tool call Claude generates adheres exactly to the defined input schema. This eliminates runtime errors caused by Claude passing `null` for a required field or a string where an integer is expected. It is a pre-call guarantee, not a post-call validation.


A validation retry loop is running but Claude keeps making the same schema error repeatedly across retries. What is most likely causing this?
A.The model needs to be replaced with a larger, more capable one for this task
B.Error feedback is too generic — it lacks the specific field name and expected type
C.The JSON schema is too complex for Claude to follow reliably in this context
D.Validation retry loops are only effective for the first retry attempt per field in each extraction
✓ Correct — Explanation
For self-correction to work, the error feedback must be specific. "Validation failed" is useless. "Field `publish_date` must be ISO 8601 format (YYYY-MM-DD). You returned `March 15, 2026` — please use `2026-03-15` instead." is actionable. The more precisely the error identifies what failed and what the correct format is, the more reliably Claude self-corrects.


A content moderation pipeline classifies 1 million items per day. The schema has 12 fields with nested objects. Validation shows 3% of outputs fail the schema on the first pass. The retry loop fixes 97% of failures. What is the remaining architectural concern?
A.A 3% initial failure rate is too high and the schema must be simplified
B.The retry loop cost (3% × full-price retries vs. Batches pricing) and retry latency must be factored into the SLA and cost model for production
C.Validation retry loops cannot be used with the Batches API
D.The remaining 3% after retry (0.09% total) can be ignored at this scale
✓ Correct — Explanation
At 1M items/day, 3% initial failure = 30,000 retry calls. If retries use synchronous API (can't use Batches for time-sensitive retries), the cost and latency model changes significantly. Additionally, the 3% of Batches failures that enter retry queues may violate latency SLAs if the pipeline has hard deadlines. Production architecture must account for the retry cost and latency of the failure-and-recover path, not just the happy path.


A structured data extraction pipeline processes legal contracts. 15% of contracts use non-standard date formats that cause schema validation failures. The retry loop sends back "Invalid date format." How should the retry prompt be improved?
A.Increase max_retries from 3 to 10
B.Remove the date field from the schema and process dates separately
C.Include in the retry: the exact field that failed, the expected format (ISO 8601: YYYY-MM-DD), the value Claude returned, and an example conversion
D.Switch to a few-shot approach with date format examples in the initial prompt
✓ Correct — Explanation
Specific error feedback is the key to reliable self-correction. The retry should include: "Field `contract_date` requires ISO 8601 format (YYYY-MM-DD). You returned `15th March 2026`. The correct value is `2026-03-15`." This gives Claude the exact field, the exact format requirement, the exact wrong value, and the correct form — all the information needed to fix it on the next attempt.


Your structured data extraction pipeline outputs a confidence score alongside each extracted field. A teammate argues these per-field confidence scores should never be used to route review decisions because "model confidence scores are unreliable." How do you respond?
A.Agree — self-reported confidence scores are always unreliable and should be removed from the schema
B.Per-field confidence scores can be a valid routing signal when calibrated against a labeled validation set: measure what confidence threshold corresponds to acceptable accuracy on labeled data, then route fields below that threshold to human review
C.Use the scores only for fields that are not critical to downstream decisions
D.Replace confidence scores with a binary `is_certain: boolean` flag to avoid the calibration problem
✓ Correct — Explanation
The distinction is calibration. Raw, uncalibrated confidence scores correlate poorly with accuracy — models can be highly confident on wrong extractions. However, when you build a labeled validation set (e.g., 500 manually verified extractions), measure actual accuracy at each confidence level, and find that confidence > 0.87 corresponds to 95% accuracy in your specific domain, you have a calibrated threshold. Fields below that threshold are routed to human review. This is different from using raw confidence as a blanket routing signal (the anti-pattern) — it is an empirically validated threshold specific to your pipeline. Removing confidence scores entirely (A) discards a useful signal. Binary flags (D) are less informative.


You are extracting data from insurance claims forms. The schema requires a `claim_type` field with enum values: `["medical", "dental", "vision", "mental_health", "pharmacy"]`. Validation shows Claude is occasionally returning `"Mental Health"` (capitalized, with space) instead of `"mental_health"`. Which two changes together guarantee this never happens?
A.Add "use lowercase with underscores" to the prompt and add a few-shot example of the correct enum value
B.Set `strict: true` on the tool definition (schema validation guaranteed) AND include the enum in the JSON schema definition — any response deviating from the enum values will be rejected before it reaches your validation code
C.Add a post-processing step that normalizes strings to the expected enum format
D.Use a different model that has better enum adherence
✓ Correct — Explanation
Two layers of guarantee: (1) defining the enum in the JSON schema tells Claude exactly which values are valid, (2) `strict: true` guarantees that the response matches the schema — any deviation (capitalization, spacing) is caught at the API level before your code processes it. Post-processing normalization is a workaround that treats the symptom; schema enforcement prevents the problem at the source.



Your extraction pipeline retries failed extractions up to 5 times. For a batch of 200 contracts, 30 contracts fail validation on every retry. Investigation shows these 30 contracts reference an addendum document ("Exhibit A") that was not included in the original document set. What should you conclude about the retries?
A.Increase the retry limit to 10 — 5 retries may not be enough for complex documents
B.Retries are ineffective when required information is absent from the source document; these 30 contracts need the missing Exhibit A documents, not more retries
C.Change the retry strategy to use a higher temperature on each retry to encourage the model to try different extraction approaches
D.Add a fallback that extracts whatever is available and marks the missing fields as "pending"
✓ Correct — Explanation
Retry-with-error-feedback is effective for format mismatches and structural errors — cases where the information exists but the model extracted it incorrectly. When required information is simply absent from the source document (the addendum exists but wasn't provided), no number of retries will help. The model cannot extract what isn't there. The correct resolution is to obtain the missing Exhibit A documents and reprocess. More retries (A) waste API calls. Higher temperature (C) increases randomness, making hallucination more likely. Option D produces incomplete data that may be misinterpreted downstream.


The code review component works iteratively: Claude analyzes a changed file, then may request related files (imports, base classes, tests) via tool calling to understand context before providing final feedback. Your application defines a tool that lets Claude request file contents; Claude invokes this tool, receives results, and continues its analysis. You're evaluating batch processing to reduce API costs. What is the primary technical constraint when considering batch processing for this workflow?
A.The batch API does not support tool definitions in request parameters.
B.Batch processing latency of up to 24 hours is too slow for pull request feedback, though the workflow could otherwise function.
C.The asynchronous model prevents executing tools mid-request and returning results for Claude to continue analysis.
D.Batch processing lacks request correlation identifiers for matching outputs to input requests.
✗ Incorrect — Explanation
The Batches API uses a fire-and-forget asynchronous model: you submit a batch and poll for completion; there is no mechanism to intercept a tool call mid-request, execute it server-side, and return the result so Claude can continue. Iterative tool-calling requires multiple round-trips within a single logical interaction — Claude calls a tool, your code executes it and returns the result, Claude resumes. This architecture is fundamentally incompatible with the batch model. The latency concern (B) is real but secondary — even if latency were acceptable, the workflow would still be broken because the tool calls cannot be served. Tool definitions (A) are supported in batch requests. Correlation IDs (D) are handled via the custom_id field.


You need to extract structured data from 50,000 product descriptions overnight. The downstream system ingests a batch file at 6 AM the next day. Which API approach is most appropriate?
A.Message Batches API — 50% cost savings with a 24-hour processing window
B.Standard synchronous API with 50 parallel threads for maximum throughput
C.Streaming API for real-time processing with minimal per-request overhead
D.Standard API with prompt caching for repeated schema definition prefixes
✓ Correct — Explanation
Overnight bulk processing with a next-morning deadline is the canonical Message Batches API use case: (1) latency-tolerant (deadline is tomorrow morning, not now), (2) non-interactive (no user waiting), (3) high volume (50k requests). The 50% cost savings on this volume is significant. Parallel synchronous requests would work but cost double and add infrastructure complexity.


Your team wants to reduce API costs for automated analysis. Currently, real-time Claude calls power two workflows: (1) a blocking pre-merge check that must complete before developers can merge, and (2) a technical debt report generated overnight for review the next morning. Your manager proposes switching both to the Message Batches API for its 50% cost savings. How should you evaluate this proposal?
A.Use batch processing for the technical debt reports only; keep real-time calls for pre-merge checks
B.Switch both workflows to batch processing with status polling to check for completion
C.Keep real-time calls for both workflows to avoid batch result ordering issues
D.Switch both to batch processing with a timeout fallback to real-time if batches take too long
✓ Correct — Explanation
The Message Batches API offers 50% cost savings but has processing times up to 24 hours with no guaranteed latency SLA. This makes it unsuitable for blocking pre-merge checks where developers wait for results, but ideal for overnight batch jobs like technical debt reports. Option B is wrong because relying on polling isn't acceptable for blocking workflows with developer wait time. Option C reflects a misconception — batch results can be correlated using custom_id fields. Option D adds unnecessary complexity when the simpler solution is matching each API to its appropriate use case.



Your CI/CD system performs three types of Claude-powered analysis: (1) quick style checks on each PR that block merging until complete, (2) comprehensive security audits of the entire codebase run weekly, and (3) test case generation triggered nightly for recently-modified modules. The Message Batches API offers 50% cost savings but can take up to 24 hours to process. You want to optimize API costs while maintaining acceptable developer experience. Which combination correctly matches each task to its API approach?
A.Use synchronous calls for PR style checks and nightly test generation; use Message Batches API only for weekly security audits.
B.Use the Message Batches API for all three tasks to maximize the 50% cost savings, and configure the pipeline to poll for batch completion.
C.Use synchronous calls for all three tasks for consistent response times, and rely on prompt caching to reduce costs across all workloads.
D.Use synchronous calls for PR style checks; use the Message Batches API for weekly security audits and nightly test generation.
✓ Correct — Explanation
PR style checks block developers from merging and require immediate responses — they must use synchronous calls. Weekly security audits and nightly test generation are scheduled tasks with flexible timelines: they already run asynchronously and can easily tolerate the up-to-24-hour batch processing window. Using the Batches API for both scheduled workflows captures the 50% cost savings on the two highest-volume workloads without degrading developer experience. Option A incorrectly uses sync for nightly test generation, missing cost savings. Option B would make PR style checks unusable. Option C foregoes all batch savings.



A batch job uses the Message Batches API to process 10,000 support ticket classifications. The job starts at 11 PM and the results are needed by 9 AM. Should you use the Batches API? What is the key risk?
A.Yes — 10 hours is sufficient buffer. Key risk: the 24-hour window means results may not be ready within 10 hours
B.No — use synchronous API instead
C.Yes — the 24-hour window guarantees completion within 9 hours
D.No — Batches API is limited to 1,000 requests
✓ Correct — Explanation
The Message Batches API has a 24-hour *window* — not a 10-hour guarantee. While most batch jobs complete much faster than 24 hours, there is no SLA. If the 9 AM deadline is hard, there is a risk of results not being ready. For guaranteed timing, use the synchronous API. This is the core Batches API tradeoff the exam tests: cost savings vs. latency unpredictability.


What is the difference between tool_choice: "auto", tool_choice: "any", and forced tool selection in the Claude API?
A."auto" forces a tool call; "any" lets Claude choose which; forced names a tool
B."auto" lets Claude decide freely; "any" guarantees some tool; forced names one
C."auto" and "any" are identical; forced is the only option that changes behavior
D."any" and forced both guarantee a call; "auto" disables tool calling entirely
✓ Correct — Explanation
tool_choice: "auto" (default) allows Claude to either call a tool or return conversational text — no guarantee of tool use. tool_choice: "any" guarantees Claude will call at least one tool but lets it choose which from available tools. Forced tool selection ({"type": "tool", "name": "..."}) guarantees a specific named tool is called. Use "any" when you need structured output but have multiple valid schemas; use forced selection when a specific tool must run first (e.g., metadata extraction before enrichment).


You are building a multi-pass document review system: first pass extracts all claims, second pass verifies each claim against sources, third pass generates a credibility report. What architectural pattern does this represent?
A.A standard validation retry loop that corrects schema errors on each pass
B.Parallel decomposition where all dimensions are analyzed simultaneously
C.Chain-of-thought prompting applied across multiple sequential API calls
D.Multi-pass review — separate specialized passes for each review dimension
✗ Incorrect — Explanation
Multi-pass review architecture dedicates each pass to a single review dimension: extract claims (no judgment, just extraction), verify claims (no extraction, just verification), report credibility (no extraction or verification, just synthesis). Each pass gets full attention on its specific task. Attempting all three in one pass causes attention dilution and lower quality on each dimension.


You need to build a contract analysis pipeline that extracts 15 fields from legal documents including parties, dates, payment terms, termination clauses, and governing law. Some documents are 80+ pages. What is the most reliable architecture?
A.Send the entire 80-page document in one API call with all 15 fields in the schema
B.Use a multi-pass architecture: pass 1 extracts structural elements (parties, dates), pass 2 extracts financial terms, pass 3 extracts legal clauses — each pass with focused schema and relevant document sections
C.Use a larger context window model and extract all 15 fields simultaneously
D.Chain 15 separate API calls, one per field
✓ Correct — Explanation
Multi-pass extraction on long documents produces significantly better results than single-pass. Each pass focuses on a subset of related fields and the relevant document sections (e.g., pass 1 only reads the header and signature blocks; pass 2 reads the payment section). This prevents attention dilution, reduces the extraction schema complexity per pass, and allows each pass to use the most relevant document context.


A team generates code with Claude and then uses the same Claude session to review that code for bugs. The review consistently misses subtle logical errors — issues that human reviewers catch immediately. What is the most likely explanation and best fix?
A.The model is running out of context space; the fix is to use /compact before the code review step
B.The generating session retains reasoning context, making it less likely to question itself
C.The same model cannot both generate and review code — use a different model instead
D.Add a "be critical" system prompt instruction to Claude before starting the review
✗ Incorrect — Explanation
Self-review in the same session is a known limitation: the model retains reasoning context from generation and is less likely to question its own decisions. This is not a model capability issue — it's a context contamination issue. An independent review instance (fresh session without the generation context) is substantially more effective at catching subtle issues. This is equivalent to the human practice of having code reviewed by someone who didn't write it. /compact (A) reduces tokens but doesn't remove the reasoning context. Different model (C) may help but misidentifies the root cause. System prompt instructions (D) are probabilistic and don't address context contamination.


Your automated code review averages 15 findings per pull request, with developers reporting a 40% false positive rate. The bottleneck is investigation time: developers must click into each finding to read Claude's reasoning before deciding whether to address or dismiss it. Your CLAUDE.md already contains comprehensive rules for acceptable patterns, and stakeholders have rejected any approach that filters findings before developer review. What change would best address the investigation time bottleneck?
A.Require Claude to include its reasoning and confidence assessment inline with each finding
B.Categorize findings as "blocking issues" versus "suggestions" with tiered review requirements
C.Add a post-processor that analyzes finding patterns and automatically suppresses those matching historical false positive signatures
D.Configure Claude to only surface findings it assesses as high confidence, filtering out uncertain flags before developers see them
✓ Correct — Explanation
The bottleneck is clicking into findings to read reasoning. Including reasoning and confidence inline with each finding eliminates that click — developers can triage at a glance without navigating away. This respects the no-filtering constraint because all findings remain visible; it simply makes the information needed for triage immediately accessible. Categorizing (B) adds a useful label but still requires developers to evaluate each finding without the reasoning they need. The post-processor (C) was explicitly rejected by stakeholders as a filtering approach. High-confidence filtering (D) was also explicitly rejected — it hides findings from developers.


Your document processing pipeline must return results within 30 hours of document receipt. You plan to use the Message Batches API (24-hour processing window). What is the maximum allowable time between document receipt and batch submission to guarantee the 30-hour SLA?
A.12 hours — leave buffer on both sides of the 24-hour window
B.24 hours — the batch window starts after submission so documents can wait a full day
C.6 hours — SLA (30h) minus the maximum batch processing time (24h) leaves 6 hours for pre-submission queuing
D.There is no safe window; the Batches API cannot guarantee a 30-hour SLA
✓ Correct — Explanation
The calculation is straightforward: SLA deadline (30h) − maximum batch processing time (24h) = maximum pre-submission delay (6h). Documents must be submitted to a batch within 6 hours of receipt to guarantee that even a worst-case 24-hour processing window still meets the 30-hour SLA. Leaving 12 hours (A) is unnecessarily conservative — documents received 7-12 hours ago would be rejected when they could safely be included. Waiting 24 hours (B) guarantees SLA violations for documents that take the full 24-hour processing window. Option D is incorrect — the Batches API can meet this SLA if submission timing is managed.


What is the primary advantage of requesting JSON output with a defined schema over free-form text output?
A.Schema validation guarantees structure and enables reliable downstream parsing
B.JSON output is always shorter and cheaper than equivalent free-form text output
C.Claude produces higher quality content when asked to output in JSON format
D.JSON format prevents Claude from making factual errors in the extracted data
✗ Incorrect — Explanation
Structured JSON output with a defined schema enables reliable programmatic processing: downstream systems can parse fields by name, validate types, and fail fast on schema violations. Free-form text requires fragile regex or NLP parsing. When combined with `strict: true`, the schema contract is guaranteed at the API level.








